# Taller EDA - Fase 1: Dataset Adult (Census Income)

**Materia:** Introducción a la Inteligencia Artificial  
**Dataset:** Adult / Census Income (UCI Repository)  
**Objetivo:** Predecir si una persona gana más o menos de 50K USD al año.

---
### Orden del preprocesamiento y por qué importa

El flujo de este notebook sigue un orden deliberado para evitar **data leakage**:

| Paso | Sobre qué datos | Razón |
|------|----------------|-------|
| Cargar y unir | Todo el dataset | Solo lectura, no aprende nada |
| Marcar `'?'` como NaN | Todo el dataset | Solo marca, no decide nada |
| EDA gráfico | Todo el dataset | Análisis exploratorio, no modifica |
| Codificar `income` (0/1) | Todo el dataset | Variable objetivo, no hay fuga posible |
| Dummificación | Todo el dataset | Excepción justificada: garantiza columnas idénticas en train y test |
| **SPLIT 80/20** | — | Punto de separación definitivo |
| Imputar nulos | Fit en train → apply en test | La moda se aprende solo del train |
| Winsorizar outliers | Fit en train → apply en test | Los límites IQR se calculan solo del train |
| Normalizar | Fit en train → apply en test | Media y std se aprenden solo del train |

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
SEED = 42

## 2. Cargar el dataset

Se unen `adult.data` y `adult.test` en un solo DataFrame.  
Esta unión no modifica ni aprende nada de los datos, solo los consolida para el EDA.

In [ ]:
columnas = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

url_train = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
url_test  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test'

df_train = pd.read_csv(url_train, names=columnas, sep=',', skipinitialspace=True)

# adult.test tiene una línea basura al inicio y puntos al final de income
df_test  = pd.read_csv(url_test, names=columnas, sep=',', skipinitialspace=True, skiprows=1)
df_test['income'] = df_test['income'].str.replace('.', '', regex=False)

df = pd.concat([df_train, df_test], ignore_index=True)

print(f'Tamaño total: {df.shape}')
df.head()

## 3. Información básica del dataset

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## 4. Marcar valores faltantes

Los `'?'` se reemplazan por `NaN` para que pandas los reconozca.  
Esto no imputa ni aprende nada — solo cambia la representación.

In [ ]:
# Solo marcamos — no imputamos todavía
df = df.replace('?', np.nan)

nulos = df.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0])
print(f'\nPorcentaje de filas con al menos un nulo: {df.isnull().any(axis=1).mean()*100:.2f}%')

## 5. Análisis gráfico (EDA)

El EDA se hace sobre el dataset completo antes del split para tener la imagen real de los datos.  
Las decisiones de preprocesamiento que surjan aquí se aplicarán **después** del split.

### 5.1 Distribución de la variable objetivo (income)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='income', palette='Set2')
plt.title('Distribución de la variable objetivo (income)')
plt.ylabel('Cantidad de registros')
plt.savefig('income_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(df['income'].value_counts(normalize=True) * 100)

### 5.2 Distribución de variables numéricas

In [ ]:
variables_numericas = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

df[variables_numericas].hist(bins=30, figsize=(14, 10), color='steelblue', edgecolor='black')
plt.suptitle('Distribución de variables numéricas', fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Boxplots (detección visual de outliers)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), variables_numericas):
    sns.boxplot(data=df, y=col, ax=ax, color='salmon')
    ax.set_title(col)
plt.tight_layout()
plt.show()

### 5.4 Variables categóricas vs income

In [ ]:
categoricas_a_revisar = ['education', 'marital-status', 'occupation', 'sex', 'race', 'workclass']

for col in categoricas_a_revisar:
    plt.figure(figsize=(12, 4))
    sns.countplot(data=df, y=col, hue='income', palette='Set2',
                  order=df[col].value_counts().index)
    plt.title(f'{col} vs income')
    plt.tight_layout()
    plt.show()

### 5.5 Matriz de correlación

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[variables_numericas].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlación - variables numéricas')
plt.show()

## 6. Codificar la variable objetivo

Se convierte `income` a 0/1 **antes** del split porque es la variable objetivo,  
no una variable predictora: codificarla no introduce ninguna fuga de información.

In [ ]:
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
print(df['income'].value_counts())

## 7. Dummificación (excepción al orden general)

La dummificación es el único paso que se aplica al dataset completo **antes** del split.  
El motivo es técnico: si se hace por separado en train y test, una categoría rara que solo
aparezca en uno de los dos generaría columnas distintas, haciendo que el modelo falle al predecir.
Aplicarla sobre el dataset completo garantiza que ambos subconjuntos tengan exactamente
las mismas columnas. Esta es una excepción reconocida y justificada en la práctica.

In [ ]:
# education es redundante con education-num → eliminar
df = df.drop(columns=['education'])

cat_cols = ['workclass', 'marital-status', 'occupation', 'relationship',
            'race', 'sex', 'native-country']

# drop_first=True evita multicolinealidad perfecta
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f'Forma después de dummificar: {df.shape}')
df.head()

## 8. SPLIT 80/20 — punto de separación definitivo

A partir de aquí el test no vuelve a participar en ningún cálculo de parámetros.  
`stratify=y` preserva la proporción 76/24 en ambos subconjuntos.

In [ ]:
X = df.drop(columns=['income'])
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'\nProporción de clases en train:\n{y_train.value_counts(normalize=True).round(3)}')
print(f'\nProporción de clases en test:\n{y_test.value_counts(normalize=True).round(3)}')

## 9. Imputación de nulos — fit en train, apply en test

La moda de cada columna se calcula **solo** con `X_train`.  
Esa misma moda se usa para rellenar los nulos en `X_test`, sin recalcularla.

In [ ]:
# Las columnas con nulos son las dummies originadas de workclass, occupation y native-country
# Después de dummificar no quedan NaN en categóricas, pero por si acaso verificamos
print('Nulos en X_train antes de imputar:')
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# Imputar columnas numéricas con nulos si existieran (precaución general)
num_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

modas_train = {}
for col in X_train.columns:
    if X_train[col].isnull().any():
        moda = X_train[col].mode()[0]      # aprende solo del train
        modas_train[col] = moda
        X_train[col] = X_train[col].fillna(moda)
        X_test[col]  = X_test[col].fillna(moda)   # aplica sin recalcular

print('Nulos tras imputar - train:', X_train.isnull().sum().sum())
print('Nulos tras imputar - test: ', X_test.isnull().sum().sum())

## 10. Winsorización de outliers — fit en train, apply en test

Los límites IQR se calculan **solo** con `X_train`.  
Esos mismos límites se usan para recortar los valores extremos en `X_test`.

In [ ]:
limites_iqr = {}

for col in num_cols:
    Q1  = X_train[col].quantile(0.25)
    Q3  = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    limites_iqr[col] = (lim_inf, lim_sup)

# Solo winsorizamos age y hours-per-week (capital-gain/loss y fnlwgt tienen extremos válidos)
for col in ['age', 'hours-per-week']:
    low, high = limites_iqr[col]
    X_train[col] = X_train[col].clip(lower=low, upper=high)  # límites del train
    X_test[col]  = X_test[col].clip(lower=low, upper=high)   # mismos límites en test

print('Winsorización aplicada con límites del train.')
print('Filas conservadas — train:', len(X_train), ' | test:', len(X_test))

## 11. Normalización — fit en train, apply en test

`fit_transform` sobre `X_train`: aprende media y std de ese subconjunto.  
`transform` sobre `X_test`: aplica los parámetros del train sin recalcularlos.

In [ ]:
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])  # aprende del train
X_test[num_cols]  = scaler.transform(X_test[num_cols])       # aplica sin reaprender

print('Train (media ≈ 0, std ≈ 1):')
print(X_train[num_cols].describe().round(3))
print('\nTest (ligeras diferencias esperadas — los parámetros son del train):')
print(X_test[num_cols].describe().round(3))

## 12. Guardar los conjuntos finales

In [ ]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv',  index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv',  index=False)

print('Archivos guardados: X_train.csv, X_test.csv, y_train.csv, y_test.csv')